# Sentiment Analysis with Hugging Face — Starter Notebook

This is the **unfinished / student version** of the notebook for the workshop.

In this notebook, you will learn how to:

1. Load a text dataset from Hugging Face.
2. Explore labels and class imbalance.
3. Tokenize text for a Transformer model.
4. Fine-tune a pretrained model for text classification.
5. Push the trained model to the Hugging Face Hub.
6. Use the trained model with the `pipeline` API.

> **Workshop note:** Some cells contain `TODO` sections. Complete them during the session.

## Step 1 — Environment Setup

Install the required libraries. In Colab, run this cell once at the beginning.

In [ ]:
!pip install -q -U datasets huggingface_hub fsspec "transformers>=4.41.0,<5.0.0" evaluate

## Step 2 — Load and Explore the Dataset

We will use a sentiment/emotion-related text dataset from Hugging Face.

For this workshop version, we use:

```python
dair-ai/emotion
```

This dataset contains English text samples labeled with emotion categories such as sadness, joy, love, anger, fear, and surprise.

In [ ]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

# TODO 1: Load the dataset from Hugging Face.
# Hint: use load_dataset("dair-ai/emotion", trust_remote_code=True)

emotions = load_dataset(_____, trust_remote_code=True)

display(emotions)

In [ ]:
# TODO 2: Convert the training split to a pandas DataFrame.
# Hint: emotions["train"].to_pandas()

df = _____

display(df.head())

### Label Mapping

The dataset stores labels as integers. For interpretation, we convert them into readable class names.

In [ ]:
# TODO 3: Complete the function that maps integer labels to label names.
# Hint: emotions["train"].features["label"].int2str(...)

def int2str(row):
    return emotions["train"].features["label"].int2str(_____)

df["label_name"] = df.apply(int2str, axis=1)

display(df.head())

### Understanding Class Imbalance

Before training a model, it is important to check whether all classes are represented equally.

Class imbalance can affect model performance, especially for minority classes.

In [ ]:
# TODO 4: Plot the label distribution.
# Hint: use value_counts() on df["label_name"]

df[_____].value_counts(ascending=True).plot(
    kind="barh",
    title="Emotion Distribution"
)

plt.xlabel("Number of Examples")
plt.show()

In [ ]:
# TODO 5: Print the exact number of samples per label.

label_counts = df[_____].value_counts()
display(label_counts)

> **Discussion Question:**  
> Which emotion labels are frequent? Which ones are rare?  
> How could this affect the model's predictions?

## Step 3 — Tokenization

Transformer models cannot directly process raw text.  
We need to convert text into numerical tokens using a tokenizer.

We will use the tokenizer that matches the pretrained model.

In [ ]:
from transformers import AutoTokenizer

# TODO 6: Choose the pretrained model checkpoint.
# Hint: "distilbert-base-uncased"

model_ckpt = _____

# TODO 7: Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(_____)

tokenizer

In [ ]:
# TODO 8: Complete the tokenize function.
# Hint: tokenize examples["text"] and use truncation=True

def tokenize(examples):
    return tokenizer(_____, truncation=True)

# TODO 9: Apply tokenization to the full dataset using map().
# Hint: emotions.map(tokenize, batched=True, batch_size=None)

emotions_encoded = emotions.map(_____, batched=True, batch_size=None)

display(emotions_encoded)

## Step 4 — Prepare the Model

Now we load a pretrained DistilBERT model and add a classification head for our emotion labels.

In [ ]:
from transformers import TFAutoModelForSequenceClassification

# Label mappings for the emotion dataset
id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

label2id = {v: k for k, v in id2label.items()}

# TODO 10: Load a TensorFlow sequence classification model.
# Hint:
# TFAutoModelForSequenceClassification.from_pretrained(
#     model_ckpt,
#     num_labels=6,
#     id2label=id2label,
#     label2id=label2id,
#     from_pt=True
# )

model = TFAutoModelForSequenceClassification.from_pretrained(
    _____,
    num_labels=_____,
    id2label=_____,
    label2id=_____,
    from_pt=True
)

model.summary()

## Step 5 — Prepare TensorFlow Datasets

The model expects batches of tokenized inputs.  
We use a data collator to dynamically pad sequences in each batch.

In [ ]:
from transformers import DataCollatorWithPadding
import tensorflow as tf

data_collator_np = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="np"
)

def custom_collate_fn(features):
    batch_np = data_collator_np(features)
    batch_tf = {k: tf.convert_to_tensor(v) for k, v in batch_np.items()}
    return batch_tf

# TODO 11: Convert the training split to a TensorFlow dataset.
# Hint: use emotions_encoded["train"].to_tf_dataset(...)

tf_train_set = emotions_encoded["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=True,
    batch_size=_____,
    collate_fn=custom_collate_fn
)

# TODO 12: Convert the validation split to a TensorFlow dataset.

tf_validation_set = emotions_encoded["validation"].to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=_____,
    collate_fn=custom_collate_fn
)

## Step 6 — Training Setup

We define the evaluation metric and compile the model.

For a short workshop, you may use only 1 epoch or a small subset of the dataset to save time.

In [ ]:
import evaluate
import numpy as np
from transformers.keras_callbacks import KerasMetricCallback

accuracy = evaluate.load("accuracy")

# TODO 13: Complete the metric function.
# Hint: predictions should be converted to class IDs using np.argmax(..., axis=1)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = _____
    return accuracy.compute(predictions=predictions, references=labels)

# TODO 14: Choose a learning rate.
optimizer = tf.keras.optimizers.Adam(learning_rate=_____)

model.compile(optimizer=optimizer)

metric_callback = KerasMetricCallback(
    metric_fn=compute_metrics,
    eval_dataset=tf_validation_set
)

## Step 7 — Train the Model

Training may take time depending on the runtime environment.

> For the live workshop, keep `epochs=1` unless you have enough GPU time.

In [ ]:
# TODO 15: Train the model.
# Hint: model.fit(...)

history = model.fit(
    x=_____,
    validation_data=_____,
    epochs=_____,
    callbacks=[metric_callback]
)

## Step 8 — Push to the Hugging Face Hub

This step is optional during the workshop.

To push your model, you need a Hugging Face account and a token with write permission.

In [ ]:
from huggingface_hub import notebook_login

# TODO 16: Login to Hugging Face.
# Run this cell and paste your token when prompted.

# notebook_login()

In [ ]:
# TODO 17: Push the model and tokenizer to the Hub.
# Replace YOUR_MODEL_NAME with a meaningful name.
# Example: "emotion-analysis-distilbert-workshop"

# model.push_to_hub("YOUR_MODEL_NAME")
# tokenizer.push_to_hub("YOUR_MODEL_NAME")

## Step 9 — Inference with Pipeline

After training or loading a model from the Hub, we can use it with the `pipeline` API.

If you did not push your own model, you can use an existing Hugging Face model instead.

In [ ]:
from transformers import pipeline

# Option A: Use the local model and tokenizer from this notebook.
# TODO 18: Create a pipeline using model and tokenizer.

classifier = pipeline(
    task=_____,
    model=_____,
    tokenizer=_____
)

sample_text = "I watched a movie yesterday. It was really awesome!"

# TODO 19: Run the classifier on sample_text.
result = classifier(_____)

display(result)

In [ ]:
# TODO 20: Try your own examples.

examples = [
    "I am very excited about this workshop!",
    "I feel nervous and unsure about the exam.",
    "This is the worst day ever."
]

for text in examples:
    prediction = classifier(text)[0]
    print(f"Text: {text}")
    print(f"Prediction: {prediction['label']} | Confidence: {prediction['score']:.3f}")
    print("-" * 60)

## Reflection Questions

1. Why do we tokenize text before using a Transformer model?
2. What problem can class imbalance create?
3. Why do we use a pretrained model instead of training from scratch?
4. What is the difference between training a model and using a pipeline for inference?
5. What should we be careful about before sharing a trained model on the Hugging Face Hub?

## Key Takeaway

Hugging Face allows us to move from dataset loading to model training and deployment within a single ecosystem.

However, model outputs should always be interpreted carefully, especially when dealing with subjective categories such as emotions or sentiment.